In [1]:
import numpy as np
from scipy.stats import multivariate_normal

In [2]:
class MultivariateNormal:

    def __init__(self, mu, Sigma):
        self.mu = np.atleast_1d(mu)
        self.Sigma = np.atleast_2d(Sigma)
        self.D = self.mu.shape[0]

        self.inv_Sigma = np.linalg.inv(self.Sigma)
        self.log_det_Sigma = np.log(np.linalg.det(self.Sigma))

    def log_pdf(self, x):
        x = np.atleast_1d(x)
        diff = x - self.mu
        exponent = -0.5 * diff.T @ self.inv_Sigma @ diff

        norm_term = -0.5 * (
            self.D * np.log(2 * np.pi) + self.log_det_Sigma
        )

        return norm_term + exponent

    def pdf(self, x):
        return np.exp(self.log_pdf(x))

    def rvs(self, size=1):
        return np.random.multivariate_normal(
            mean=self.mu,
            cov=self.Sigma,
            size=size
        )

In [5]:
def main():

    np.random.seed(0)

    x = np.array([1.0, -1.0])
    mu = np.zeros(2)

    print("---- Spherical Gaussian ----")
    Sigma = 2 * np.eye(2)
    llm_model = MultivariateNormal(mu, Sigma)
    scipy_model = multivariate_normal(mean=mu, cov=Sigma)

    print("LLM PDF:", llm_model.pdf(x))
    print("SciPy PDF:", scipy_model.pdf(x))
    print("Difference:", abs(llm_model.pdf(x) - scipy_model.pdf(x)))
    print()

    print("---- Diagonal Gaussian ----")
    Sigma = np.diag([1.0, 4.0])
    llm_model = MultivariateNormal(mu, Sigma)
    scipy_model = multivariate_normal(mean=mu, cov=Sigma)

    print("LLM PDF:", llm_model.pdf(x))
    print("SciPy PDF:", scipy_model.pdf(x))
    print("Difference:", abs(llm_model.pdf(x) - scipy_model.pdf(x)))
    print()

    print("---- Full Covariance Gaussian ----")
    Sigma = np.array([[2.0, 0.5],
                      [0.5, 1.0]])
    llm_model = MultivariateNormal(mu, Sigma)
    scipy_model = multivariate_normal(mean=mu, cov=Sigma)

    print("LLM PDF:", llm_model.pdf(x))
    print("SciPy PDF:", scipy_model.pdf(x))
    print("Difference:", abs(llm_model.pdf(x) - scipy_model.pdf(x)))
    print()

    print("---- Sampling Test ----")
    samples = llm_model.rvs(size=100000)
    print("Sample mean:", np.mean(samples, axis=0))
    print("True mean:", mu)
    print()
    print("Sample covariance:\n", np.cov(samples.T))
    print("True covariance:\n", Sigma)


In [6]:
if __name__ == "__main__":
    main()

---- Spherical Gaussian ----
LLM PDF: 0.04826617631502696
SciPy PDF: 0.04826617631502696
Difference: 0.0

---- Diagonal Gaussian ----
LLM PDF: 0.04259475109761325
SciPy PDF: 0.04259475109761325
Difference: 0.0

---- Full Covariance Gaussian ----
LLM PDF: 0.038367593182524695
SciPy PDF: 0.03836759318252468
Difference: 1.3877787807814457e-17

---- Sampling Test ----
Sample mean: [-0.00235151  0.00538105]
True mean: [0. 0.]

Sample covariance:
 [[1.99572161 0.49552575]
 [0.49552575 0.99296044]]
True covariance:
 [[2.  0.5]
 [0.5 1. ]]
